<div align="center">
<p align="center" style="width: 100%;">
    <img src="https://raw.githubusercontent.com/vlm-run/.github/refs/heads/main/profile/assets/vlm-black.svg" alt="VLM Run Logo" width="80" style="margin-bottom: -5px; color: #2e3138; vertical-align: middle; padding-right: 5px;"><br>
</p>
<p align="center"><a href="https://docs.vlm.run"><b>Website</b></a> | <a href="https://docs.vlm.run/"><b>API Docs</b></a> | <a href="https://docs.vlm.run/blog"><b>Blog</b></a> | <a href="https://discord.gg/AMApC2UzVY"><b>Discord</b></a> | <a href="https://chat.vlm.run"><b>Chat</b></a>
</p>
</div>

# VLM Run Orion - Image Understanding, Reasoning and Execution (Node.js)

This comprehensive cookbook demonstrates [VLM Run Orion's](https://vlm.run/orion) image understanding, reasoning and execution capabilities using **Node.js/TypeScript**. For more details on the API, see the [Agent API docs](https://docs.vlm.run/agents/introduction).

For this notebook, we'll cover how to use the **VLM Run Agent Chat Completions API** - an OpenAI-compatible interface for building powerful visual intelligence with the same familiar chat-completions interface.

We'll cover the following topics:
 1. Image VQA (captioning, tagging, question-answering)
 2. Object Detection (people, faces, objects, etc.)
 3. Object Segmentation (semantic, instance, etc.)
 4. UI Parsing (Graphical UI parsing and understanding)
 5. OCR (text detection, recognition, and understanding)
 6. Image Generation (text-to-image, in-painting, out-painting, etc.)
 7. Image Tools (cropping, super-resolution, rotating, etc.)

## Prerequisites

- Node.js 18+
- VLM Run API key (get one at [app.vlm.run](https://app.vlm.run))
- Deno or tslab kernel for running TypeScript in Jupyter


## Setup

First, install the required packages and configure the environment.


In [8]:
// Install the VLM Run SDK
// npm install vlmrun openai zod zod-to-json-schema

// If using Deno kernel, install dependencies via npm specifiers
// For tslab, run: npm install vlmrun openai zod zod-to-json-schema in your project directory


In [9]:
// Import the VLM Run SDK and dependencies
import { VlmRun } from "npm:vlmrun";
import { z } from "npm:zod";
import { zodToJsonSchema } from "npm:zod-to-json-schema";


In [10]:
// Get API key from environment variable
const VLMRUN_API_KEY = Deno.env.get("VLMRUN_API_KEY");

if (!VLMRUN_API_KEY) {
    throw new Error("Please set the VLMRUN_API_KEY environment variable");
}

console.log("✓ API Key loaded successfully");


✓ API Key loaded successfully


## Initialize the VLM Run Client

We use the OpenAI-compatible chat completions interface through the VLM Run SDK.


In [ ]:
// Initialize the VLM Run client using the SDK
const client = new VlmRun({
    apiKey: VLMRUN_API_KEY,
    baseURL: "https://agent.vlm.run/v1"  // Use the agent API endpoint
});

console.log("✓ VLM Run client initialized successfully!");
console.log("Base URL: https://agent.vlm.run/v1");
console.log("Model: vlmrun-orion-1");


✓ VLM Run client initialized successfully!
Base URL: https://agent.vlm.run/v1
Model: vlmrun-orion-1


## Response Models (Schemas)

We define Zod schemas for structured outputs. These schemas provide type-safe, validated responses.


In [12]:
// Image URL Response Schema
const ImageUrlResponseSchema = z.object({
    url: z.string().describe("Pre-signed URL to the image")
});

type ImageUrlResponse = z.infer<typeof ImageUrlResponseSchema>;

// Image URL List Response Schema
const ImageUrlListResponseSchema = z.object({
    urls: z.array(ImageUrlResponseSchema).describe("List of pre-signed image URL responses")
});

type ImageUrlListResponse = z.infer<typeof ImageUrlListResponseSchema>;

// Detection Schema
const DetectionSchema = z.object({
    label: z.string().describe("Name of the detected object"),
    xywh: z.tuple([z.number(), z.number(), z.number(), z.number()])
        .describe("Bounding box (x, y, width, height) normalized from 0-1"),
    confidence: z.number().nullable().optional().describe("Detection confidence score from 0-1")
});

// Detections Response Schema
const DetectionsResponseSchema = z.object({
    detections: z.array(DetectionSchema).describe("List of detected objects with bounding boxes")
});

type DetectionsResponse = z.infer<typeof DetectionsResponseSchema>;

// Keypoint Schema
const KeypointSchema = z.object({
    xy: z.tuple([z.number(), z.number()])
        .describe("Normalized keypoint coordinates (x, y) between 0-1"),
    label: z.string().describe("Label of the keypoint")
});

// Keypoints Response Schema
const KeypointsResponseSchema = z.object({
    keypoints: z.array(KeypointSchema).describe("List of detected keypoints")
});

type KeypointsResponse = z.infer<typeof KeypointsResponseSchema>;

console.log("✓ Response schemas defined successfully!");
console.log("Schemas include type-safe validation for structured outputs.");


✓ Response schemas defined successfully!
Schemas include type-safe validation for structured outputs.


## Helper Functions

We create helper functions to simplify making chat completion requests with structured outputs.


In [13]:
/**
 * Make a chat completion request with optional images and structured output.
 * 
 * @param prompt - The text prompt/instruction
 * @param images - Optional list of images to process (URLs)
 * @param responseSchema - Optional Zod schema for structured output
 * @param model - Model to use (default: vlmrun-orion-1:auto)
 * @returns Parsed response if responseSchema provided, else raw response text
 */
async function chatCompletion<T>(
    prompt: string,
    images?: string[],
    responseSchema?: z.ZodSchema<T>,
    model: string = "vlmrun-orion-1:auto"
): Promise<T | string> {
    const content: any[] = [];
    content.push({ type: "text", text: prompt });

    if (images) {
        for (const image of images) {
            if (typeof image === "string") {
                if (!image.startsWith("http")) {
                    throw new Error("Image URLs must start with http or https");
                }
                content.push({
                    type: "image_url",
                    image_url: { url: image, detail: "auto" }
                });
            }
        }
    }

    const kwargs: any = {
        model: model,
        messages: [{ role: "user", content: content }]
    };

    if (responseSchema) {
        kwargs.response_format = {
            type: "json_schema",
            schema: zodToJsonSchema(responseSchema)
        } as any;
    }

    const response = await client.agent.completions.create(kwargs);
    const responseText = response.choices[0].message.content || "";

    if (responseSchema) {
        const parsed = JSON.parse(responseText);
        return responseSchema.parse(parsed) as T;
    }

    return responseText;
}

console.log("✓ Helper functions defined!");


✓ Helper functions defined!


## Image Understanding, Reasoning, and Execution Capabilities

VLM Run agents can perform a wide range of image processing tasks including object detection, face detection, segmentation, OCR, and more.


### 1. Captioning & Tagging

The simplest operation - load an image from a URL and caption it.


In [14]:
const IMAGE_URL = "https://storage.googleapis.com/vlm-data-public-prod/hub/examples/image.caption/car.jpg";

const result = await chatCompletion(
    "Generate a detailed description of this image.",
    [IMAGE_URL]
);

console.log(">> RESPONSE");
console.log(result);
console.log("\n>> IMAGE URL:", IMAGE_URL);


>> RESPONSE
The image `img_dbdb04` features a vintage, light teal or mint green Volkswagen Beetle parked on a paved street. The car is shown in a full side profile, facing right, and features classic chrome hubcaps, a dark running board along its lower side, and a subtle white stripe below the rear window. 

In the background, there is a light yellow or tan building wall with two wooden doors: a double door with an arched top to the left of the car's roof and a single, taller door with a white frame to the right of the car's front. The ground is paved with rectangular cobblestones or bricks, contributing to a charming, nostalgic scene.

>> IMAGE URL: https://storage.googleapis.com/vlm-data-public-prod/hub/examples/image.caption/car.jpg


### 2a. Object Detection

Detect objects in images with bounding boxes. The agent can detect common objects like people, vehicles, animals, and more.


In [15]:
const IMAGE_URL = "https://storage.googleapis.com/vlm-data-public-prod/hub/examples/image.agent/10-finding-nemo.jpeg";

const result = await chatCompletion(
    "Detect all the sea creatures in this image",
    [IMAGE_URL],
    DetectionsResponseSchema
) as DetectionsResponse;

console.log(">> RESPONSE");
console.log(result);
console.log(`\n>> Detected ${result.detections.length} objects`);
result.detections.forEach((det, i) => {
    console.log(`  ${i + 1}. ${det.label}: xywh=[${det.xywh.map(v => v.toFixed(3)).join(", ")}]`);
});


>> RESPONSE
{
  detections: [
    {
      label: "Nemo (clownfish)",
      xywh: [ 0.267, 0.335, 0.244, 0.299 ],
      confidence: 0.98
    },
    {
      label: "Dory (blue tang)",
      xywh: [ 0.405, 0.134, 0.357, 0.5 ],
      confidence: 0.97
    },
    {
      label: "Marlin (clownfish)",
      xywh: [ 0.379, 0.64, 0.243, 0.344 ],
      confidence: 0.96
    },
    {
      label: "Crush (sea turtle)",
      xywh: [ 0.004, 0.466, 0.174, 0.505 ],
      confidence: 0.95
    },
    {
      label: "Squirt (sea turtle)",
      xywh: [ 0.802, 0.546, 0.168, 0.306 ],
      confidence: 0.93
    },
    {
      label: "Hank (octopus)",
      xywh: [ 0.589, 0.566, 0.259, 0.378 ],
      confidence: 0.82
    },
    {
      label: "Bruce (shark)",
      xywh: [ 0.11, 0.225, 0.206, 0.153 ],
      confidence: 0.92
    },
    {
      label: "Anchor (shark)",
      xywh: [ 0.013, 0.057, 0.194, 0.235 ],
      confidence: 0.91
    },
    {
      label: "Chum (shark)",
      xywh: [ 0.218, 0.068, 0.128, 

### 2b. Object Detection with Specific Prompt

You can specify exactly which objects to detect using natural language.


In [16]:
const IMAGE_URL = "https://storage.googleapis.com/vlm-data-public-prod/hub/examples/image.caption/car.jpg";

const result = await chatCompletion(
    "Detect the 'car' and its 'wheels' in the image",
    [IMAGE_URL],
    DetectionsResponseSchema
) as DetectionsResponse;

console.log(">> RESPONSE");
console.log(result);
console.log(`\n>> Detected ${result.detections.length} objects`);
result.detections.forEach((det, i) => {
    console.log(`  ${i + 1}. ${det.label}: xywh=[${det.xywh.map(v => v.toFixed(3)).join(", ")}]`);
});


>> RESPONSE
{
  detections: [
    {
      label: "car",
      xywh: [ 0.008, 0.211, 0.984, 0.6 ],
      confidence: 0.95
    },
    {
      label: "wheel",
      xywh: [ 0.542, 0.556, 0.133, 0.222 ],
      confidence: 0.9
    },
    {
      label: "wheel",
      xywh: [ 0.142, 0.556, 0.133, 0.222 ],
      confidence: 0.9
    }
  ]
}

>> Detected 3 objects
  1. car: xywh=[0.008, 0.211, 0.984, 0.600]
  2. wheel: xywh=[0.542, 0.556, 0.133, 0.222]
  3. wheel: xywh=[0.142, 0.556, 0.133, 0.222]


### 2c. Face Detection

Detect and localize faces in images with bounding boxes.


In [17]:
const IMAGE_URL = "https://storage.googleapis.com/vlm-data-public-prod/hub/examples/media.tv-news/finance_bb_3_speakers.jpg";

const result = await chatCompletion(
    "Detect all the faces in the image",
    [IMAGE_URL],
    DetectionsResponseSchema
) as DetectionsResponse;

console.log(">> RESPONSE");
console.log(result);
console.log(`\n>> Detected ${result.detections.length} faces`);
result.detections.forEach((det, i) => {
    console.log(`  Face ${i + 1}: ${det.label}, xywh=[${det.xywh.map(v => v.toFixed(3)).join(", ")}]`);
});


>> RESPONSE
{
  detections: [
    {
      label: "face",
      xywh: [ 0.063, 0.197, 0.275, 0.524 ],
      confidence: 0.98
    },
    {
      label: "face",
      xywh: [ 0.352, 0.197, 0.275, 0.524 ],
      confidence: 0.97
    },
    {
      label: "face",
      xywh: [ 0.652, 0.197, 0.275, 0.524 ],
      confidence: 0.96
    }
  ]
}

>> Detected 3 faces
  Face 1: face, xywh=[0.063, 0.197, 0.275, 0.524]
  Face 2: face, xywh=[0.352, 0.197, 0.275, 0.524]
  Face 3: face, xywh=[0.652, 0.197, 0.275, 0.524]


### 2d. Person Detection

Detect and localize people in images with bounding boxes.


In [18]:
const IMAGE_URL = "https://storage.googleapis.com/vlm-data-public-prod/hub/examples/image.agent/lunch-skyscraper.jpg";

const result = await chatCompletion(
    "Detect all the people in the image",
    [IMAGE_URL],
    DetectionsResponseSchema
) as DetectionsResponse;

console.log(">> RESPONSE");
console.log(result);
console.log(`\n>> Detected ${result.detections.length} people`);
result.detections.forEach((det, i) => {
    console.log(`  Person ${i + 1}: ${det.label}, xywh=[${det.xywh.map(v => v.toFixed(3)).join(", ")}]`);
});


>> RESPONSE
{
  detections: [
    {
      label: "person",
      xywh: [ 0.04, 0.304, 0.074, 0.24 ],
      confidence: 0.98
    },
    {
      label: "person",
      xywh: [ 0.087, 0.291, 0.084, 0.273 ],
      confidence: 0.97
    },
    {
      label: "person",
      xywh: [ 0.167, 0.287, 0.088, 0.263 ],
      confidence: 0.96
    },
    {
      label: "person",
      xywh: [ 0.232, 0.283, 0.088, 0.297 ],
      confidence: 0.95
    },
    {
      label: "person",
      xywh: [ 0.314, 0.318, 0.086, 0.28 ],
      confidence: 0.94
    },
    {
      label: "person",
      xywh: [ 0.385, 0.315, 0.089, 0.27 ],
      confidence: 0.93
    },
    {
      label: "person",
      xywh: [ 0.468, 0.3, 0.088, 0.296 ],
      confidence: 0.92
    },
    {
      label: "person",
      xywh: [ 0.55, 0.327, 0.088, 0.278 ],
      confidence: 0.91
    },
    {
      label: "person",
      xywh: [ 0.632, 0.318, 0.087, 0.297 ],
      confidence: 0.9
    },
    {
      label: "person",
      xywh: [ 0.71, 0.

### 2e. Detect and blur faces

Detect faces and blur them for privacy protection. Here we combine object / face detection with an image tool.


In [19]:
const IMAGE_URL = "https://storage.googleapis.com/vlm-data-public-prod/hub/examples/media.tv-news/finance_bb_3_speakers.jpg";

const response = await client.agent.completions.create({
    model: "vlmrun-orion-1:auto",
    messages: [{
        role: "user",
        content: [
            { type: "text", text: "Blur all the faces in this image and return the blurred image object ID" },
            { type: "image_url", image_url: { url: IMAGE_URL, detail: "auto" } }
        ]
    }],
    response_format: {
        type: "json_schema",
        schema: zodToJsonSchema(ImageUrlResponseSchema)
    }
});

const result = JSON.parse(response.choices[0].message.content || "{}");
const sessionId = (response as any).session_id;
const messageContent = response.choices[0]?.message?.content || "";
const objectId = messageContent.match(/img_[a-f0-9]{6}/)?.[0];

console.log(">> RESPONSE");
console.log(result);

if (sessionId && objectId) {
    const imageData = await client.artifacts.get({
        sessionId: sessionId,
        objectId: objectId,
        rawResponse: true
    });
    console.log(`\n>> Retrieved blurred image: ${(imageData as Buffer).length} bytes`);
}


>> RESPONSE
{ url: "img_e095e1" }

>> Retrieved blurred image: 92439 bytes


### 3. Keypoint Detection

Detect keypoints in images for counting and localization tasks.


In [20]:
const IMAGE_URL = "https://storage.googleapis.com/vlm-data-public-prod/hub/examples/image.object-detection/donuts.png";

const result = await chatCompletion(
    "Detect all the donuts as keypoints and return the coordinates.",
    [IMAGE_URL],
    KeypointsResponseSchema
) as KeypointsResponse;

console.log(">> RESPONSE");
console.log(result);
console.log(`\n>> Detected ${result.keypoints.length} keypoints`);
result.keypoints.forEach((kp, i) => {
    console.log(`  ${i + 1}. ${kp.label}: xy=[${kp.xy.map(v => v.toFixed(3)).join(", ")}]`);
});


>> RESPONSE
{
  keypoints: [
    { xy: [ 0.1094, 0.1094 ], label: "donut" },
    { xy: [ 0.3594, 0.0781 ], label: "donut" },
    { xy: [ 0.5479, 0.1094 ], label: "donut" },
    { xy: [ 0.7881, 0.1094 ], label: "donut" },
    { xy: [ 0.7881, 0.2842 ], label: "donut" },
    { xy: [ 0.5, 0.5 ], label: "donut" },
    { xy: [ 0.2656, 0.3594 ], label: "donut" },
    { xy: [ 0.0537, 0.5 ], label: "donut" },
    { xy: [ 0.0293, 0.8525 ], label: "donut" },
    { xy: [ 0.2305, 0.7441 ], label: "donut" },
    { xy: [ 0.5, 0.832 ], label: "donut" },
    { xy: [ 0.7881, 0.7441 ], label: "donut" },
    { xy: [ 0.8486, 0.9414 ], label: "donut" },
    { xy: [ 0.9639, 0.5 ], label: "donut" }
  ]
}

>> Detected 14 keypoints
  1. donut: xy=[0.109, 0.109]
  2. donut: xy=[0.359, 0.078]
  3. donut: xy=[0.548, 0.109]
  4. donut: xy=[0.788, 0.109]
  5. donut: xy=[0.788, 0.284]
  6. donut: xy=[0.500, 0.500]
  7. donut: xy=[0.266, 0.359]
  8. donut: xy=[0.054, 0.500]
  9. donut: xy=[0.029, 0.853]
  10. donut: x

### 4. Segmentation

Create pixel-level segmentation masks for objects, people or regions in images.


In [21]:
const IMAGE_URL = "https://storage.googleapis.com/vlm-data-public-prod/hub/examples/image.agent/lunch-skyscraper.jpg";

const response = await client.agent.completions.create({
    model: "vlmrun-orion-1:auto",
    messages: [{
        role: "user",
        content: [
            { type: "text", text: "Detect all the people in this image, and segment them. Return the segmented image object ID" },
            { type: "image_url", image_url: { url: IMAGE_URL, detail: "auto" } }
        ]
    }],
    response_format: {
        type: "json_schema",
        schema: zodToJsonSchema(ImageUrlResponseSchema)
    }
});

const result = JSON.parse(response.choices[0].message.content || "{}");
const sessionId = (response as any).session_id;
const messageContent = response.choices[0]?.message?.content || "";
const objectId = messageContent.match(/img_[a-f0-9]{6}/)?.[0];

console.log(">> RESPONSE");
console.log(result);

if (sessionId && objectId) {
    const imageData = await client.artifacts.get({
        sessionId: sessionId,
        objectId: objectId,
        rawResponse: true
    });
    console.log(`\n>> Retrieved segmented image: ${(imageData as Buffer).length} bytes`);
}


>> RESPONSE
{ url: "img_60f071" }

>> Retrieved segmented image: 106415 bytes


## 5. OCR (Optical Character Recognition)

Extract text from images using OCR capabilities.


In [22]:
const IMAGE_URL = "https://storage.googleapis.com/vlm-data-public-prod/hub/examples/agent_use_cases/hand_writting_beautification/image-ocr.jpg";

const result = await chatCompletion(
    "Read the text in this image",
    [IMAGE_URL]
);

console.log(">> RESPONSE");
console.log(result);
console.log("\n>> IMAGE URL:", IMAGE_URL);


>> RESPONSE
The text in the image `img_7edc1a` reads:

"Today is Thursday, October 20th- But it definitely feels like a Friday. I'm already considering making a second cup of coffee- and I haven't even finished my first. Do I have a problem?

Sometimes I'll flip through older notes I've taken, and my handwriting is unrecognizable. Perhaps it depends on the type of pen I use?

I've tried writing in all caps BUT IT LOOKS SO FORCED AND UNNATURAL

Often times, I'll just take notes on my laptop, but I still seem to gravitate toward pen and paper. Any advice on what to improve? I already feel stressed out looking back at what I've just written- it looks like 3 different people wrote this!"

>> IMAGE URL: https://storage.googleapis.com/vlm-data-public-prod/hub/examples/agent_use_cases/hand_writting_beautification/image-ocr.jpg


### 6. Image Generation

Create, modify and remix images from text prompts or existing visuals.


### 6a. Virtual Try-On

Generate a virtual try-on of a dress on a person, with unique views and a seamless compositing.


In [23]:
const DRESS_URL = "https://storage.googleapis.com/vlm-data-public-prod/hub/examples/agent_use_cases/virtual_try_on/dress.png";
const PERSON_URL = "https://storage.googleapis.com/vlm-data-public-prod/hub/examples/agent_use_cases/virtual_try_on/person.png";

console.log("Dress URL:", DRESS_URL);
console.log("Person URL:", PERSON_URL);


Dress URL: https://storage.googleapis.com/vlm-data-public-prod/hub/examples/agent_use_cases/virtual_try_on/dress.png
Person URL: https://storage.googleapis.com/vlm-data-public-prod/hub/examples/agent_use_cases/virtual_try_on/person.png


In [30]:
// Generate a virtual try-on of a dress on a person, with unique views
const response = await client.agent.completions.create({
    model: "vlmrun-orion-1:auto",
    messages: [{
        role: "user",
        content: [
            { type: "text", text: "You are provided with two images: one of a dress(the first image) and one of a person(the second image). Generate a few highly realistic virtual try-on by seamlessly compositing the dress onto the person, ensuring natural fit, alignment, and that the person appears fully and appropriately dressed. Provide 2 images (9:16 aspect ratio) as output: one from the front and one from the side. Return the image object IDs." },
            { type: "image_url", image_url: { url: DRESS_URL, detail: "auto" } },
            { type: "image_url", image_url: { url: PERSON_URL, detail: "auto" } }
        ]
    }],
    response_format: {
        type: "json_schema",
        schema: zodToJsonSchema(ImageUrlListResponseSchema)
    }
});

const result = JSON.parse(response.choices[0].message.content || "{}");
const sessionId = (response as any).session_id;
const messageContent = response.choices[0]?.message?.content || "";
const objectIds = messageContent.match(/(?:img|url|vid)_[a-f0-9]{6}/g) || [];

console.log(">> RESPONSE");
console.log(result);
console.log(`\n>> Generated ${objectIds.length} images`);

if (sessionId && objectIds.length > 0) {
    for (let i = 0; i < objectIds.length; i++) {
        const objectId = objectIds[i];
        const imageData = await client.artifacts.get({
            sessionId: sessionId,
            objectId: objectId,
            rawResponse: true
        });
        console.log(`  Image ${i + 1} (${objectId}): ${(imageData as Buffer).length} bytes`);
    }
}


>> RESPONSE
{ urls: [ { url: "img_6f614c" }, { url: "img_f5e47d" } ] }

>> Generated 2 images
  Image 1 (img_6f614c): 67284 bytes
  Image 2 (img_f5e47d): 61771 bytes


### 7. Template Matching

Find a template image within a larger reference image.


In [25]:
const TEMPLATE_URL = "https://storage.googleapis.com/vlm-data-public-prod/hub/examples/agent_use_cases/template-search/image-12.png";
const REFERENCE_URL = "https://storage.googleapis.com/vlm-data-public-prod/hub/examples/agent_use_cases/template-search/image-13.png";

console.log("Template URL:", TEMPLATE_URL);
console.log("Reference URL:", REFERENCE_URL);


Template URL: https://storage.googleapis.com/vlm-data-public-prod/hub/examples/agent_use_cases/template-search/image-12.png
Reference URL: https://storage.googleapis.com/vlm-data-public-prod/hub/examples/agent_use_cases/template-search/image-13.png


In [26]:
const result = await chatCompletion(
    "Given two images, identify the specified item from the second image within the first image. Clearly highlight and draw bounding boxes around all occurrences of the item in the first image. Provide a brief description of the results.",
    [TEMPLATE_URL, REFERENCE_URL],
    DetectionsResponseSchema
) as DetectionsResponse;

console.log(">> RESPONSE");
console.log(result);
console.log(`\n>> Found ${result.detections.length} matches`);
result.detections.forEach((det, i) => {
    console.log(`  ${i + 1}. ${det.label}: xywh=[${det.xywh.map(v => v.toFixed(3)).join(", ")}]`);
});


>> RESPONSE
{
  detections: [
    {
      label: "lemon",
      xywh: [ 0.16, 0.48, 0.08, 0.08 ],
      confidence: 0.9
    },
    {
      label: "lemon",
      xywh: [ 0.775, 0.29, 0.09, 0.08 ],
      confidence: 0.9
    }
  ]
}

>> Found 2 matches
  1. lemon: xywh=[0.160, 0.480, 0.080, 0.080]
  2. lemon: xywh=[0.775, 0.290, 0.090, 0.080]


### 8. UI Parsing

Parse user interface elements from screenshots.


In [27]:
const IMAGE_URL = "https://storage.googleapis.com/vlm-data-public-prod/hub/examples/web.ui-automation/win11.jpeg";

const result = await chatCompletion(
    "Parse the UI of this screenshot and detect all the UI elements.",
    [IMAGE_URL],
    DetectionsResponseSchema
) as DetectionsResponse;

console.log(">> RESPONSE");
console.log(`Detected ${result.detections.length} UI elements`);
result.detections.slice(0, 10).forEach((det, i) => {
    console.log(`  ${i + 1}. ${det.label}: xywh=[${det.xywh.map(v => v.toFixed(3)).join(", ")}]`);
});
if (result.detections.length > 10) {
    console.log(`  ... and ${result.detections.length - 10} more`);
}


>> RESPONSE
Detected 47 UI elements
  1. text: xywh=[0.378, 0.110, 0.033, 0.021]
  2. icon: xywh=[0.497, 0.231, 0.076, 0.119]
  3. icon: xywh=[0.287, 0.227, 0.076, 0.115]
  4. icon: xywh=[0.361, 0.346, 0.067, 0.099]
  5. icon: xywh=[0.637, 0.596, 0.053, 0.042]
  6. icon: xywh=[0.305, 0.646, 0.198, 0.082]
  7. icon: xywh=[0.305, 0.727, 0.200, 0.092]
  8. icon: xywh=[0.516, 0.649, 0.181, 0.078]
  9. icon: xywh=[0.925, 0.938, 0.068, 0.056]
  10. icon: xywh=[0.517, 0.726, 0.178, 0.091]
  ... and 37 more


### 9. Streaming Responses

For long-running tasks, you can use streaming to get partial results as they become available.


In [28]:
const IMAGE_URL = "https://storage.googleapis.com/vlm-data-public-prod/hub/examples/image.caption/car.jpg";

const stream = await client.agent.completions.create({
    model: "vlmrun-orion-1:auto",
    messages: [{
        role: "user",
        content: [
            { type: "text", text: "Describe this image in detail" },
            { type: "image_url", image_url: { url: IMAGE_URL } }
        ]
    }],
    stream: true
});

console.log("Streaming response:");
console.log("----------------------------------------");
let fullResponse = "";
for await (const chunk of stream) {
    const content = chunk.choices[0]?.delta?.content;
    if (content) {
        fullResponse += content;
        // In a real notebook, you might want to display this incrementally
        process.stdout.write(content);
    }
}
console.log("\n----------------------------------------");
console.log("\n>> Full response length:", fullResponse.length, "characters");


Streaming response:
----------------------------------------

----------------------------------------

>> Full response length: 675 characters


### 10. Using Artifacts API for Preview Images

When the API returns image URLs (like in image generation, segmentation, or filtering operations), you can use the artifacts API to retrieve the actual image data for preview, saving, or further processing.

In [29]:
// Generate a processed image and retrieve using artifacts API
const IMAGE_URL = "https://storage.googleapis.com/vlm-data-public-prod/hub/examples/image.caption/car.jpg";

const response = await client.agent.completions.create({
    model: "vlmrun-orion-1:auto",
    messages: [{
        role: "user",
        content: [
            { type: "text", text: "Apply a vintage filter to this image and return the processed image object ID" },
            { type: "image_url", image_url: { url: IMAGE_URL, detail: "auto" } }
        ]
    }],
    response_format: {
        type: "json_schema",
        schema: zodToJsonSchema(ImageUrlResponseSchema)
    }
});

const result = JSON.parse(response.choices[0].message.content || "{}");
const sessionId = (response as any).session_id;
const messageContent = response.choices[0]?.message?.content || "";
const objectId = messageContent.match(/img_[a-f0-9]{6}/)?.[0];

console.log(">> RESPONSE");
console.log(result);

if (sessionId && objectId) {
    const imageData = await client.artifacts.get({
        sessionId: sessionId,
        objectId: objectId,
        rawResponse: true
    });
    console.log(`\n>> Retrieved processed image: ${(imageData as Buffer).length} bytes`);
    console.log("\n>> The image data is now available for:");
    console.log("  - Saving to file");
    console.log("  - Displaying in notebook");
    console.log("  - Further processing with image libraries");
}


>> RESPONSE
{ url: "img_e47bc5" }

>> Retrieved processed image: 151519 bytes

>> The image data is now available for:
  - Saving to file
  - Displaying in notebook
  - Further processing with image libraries


---

## Conclusion

This cookbook demonstrated the comprehensive capabilities of the **VLM Run Orion Image Agent API** using Node.js/TypeScript.

### Key Takeaways

1. **OpenAI-Compatible Interface**: The API follows the OpenAI chat completions format, making it easy to integrate with existing workflows and tools.
2. **Structured Outputs**: Use Zod schemas with `response_format` parameter to get type-safe, validated responses with automatic parsing.
3. **Type Safety**: TypeScript and Zod provide compile-time and runtime type checking for better developer experience.
4. **Streaming Support**: For long-running tasks, enable streaming to receive partial results as they become available, improving user experience.
5. **Flexible Prompting**: Natural language prompts allow you to combine multiple operations in a single request, reducing API calls and latency.
6. **Rich Capabilities**: The API supports object detection, segmentation, OCR, image generation, UI parsing, and more.

### Next Steps

- Explore the [VLM Run Documentation](https://docs.vlm.run) for more details
- Join our [Discord community](https://discord.gg/AMApC2UzVY) for support
- Check out more examples in the [VLM Run Cookbook](https://github.com/vlm-run/vlmrun-cookbook)
- Review the [VLM Run Node.js SDK](https://github.com/vlm-run/vlmrun-node-sdk) documentation

Happy building!
